# M1-YOLO Conservative Re-train (백업 옵션)

**조건**: M1-aggressive가 ep10에서 R<0.1 또는 mAP<0.2 학습 실패 시 사용

**보수적 변경 (안정적 학습 우선):**
- 모델: yolov11m → **yolov11m 그대로** (또는 yolov11s 선택)
- imgsz: 1280 (그대로 — 소형 객체 위해)
- lr0: 5e-4 → **1e-4** (10배 줄임, 안정 수렴)
- freeze: 0 → **10** (백본 freeze, fine-tune)
- mosaic: 1.0 → **0.5** (augmentation 약화)
- mixup: 0.15 → **0.0** (제거)
- copy_paste: 0.5 → **0.3**
- epochs: 100 → **80**
- patience: 30 → **25**

**Drive 준비:**
1. `structural_compressed.zip` (이미 있음)

**예상 시간:** 8~10시간 (A100), 12~15시간 (T4)

**Resume + Drive 자동 저장**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ Drive 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect_A')  # C 계정 기준, A는 'drone_inspect'
ZIP_NAME = 'structural_compressed.zip'

WORK = Path('/content/m1_train')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음'

ds_dir = WORK / 'structural_compressed'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')

for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        print(f'  {split}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f"""path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: crack
  1: waterproof_defect
  2: caulking_defect
"""
data_yaml = WORK / 'structural.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

In [ ]:
AUTOSAVE = DRIVE_DIR / 'm1_conservative_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m1_conservative')
PROJECT.mkdir(parents=True, exist_ok=True)

RESUME = False
resume_pt = AUTOSAVE / 'last.pt'
if resume_pt.exists():
    print(f'Resume: {resume_pt}')
    weights_dir = PROJECT / 'train' / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(resume_pt, weights_dir / 'last.pt')
    if (AUTOSAVE / 'best.pt').exists():
        shutil.copy2(AUTOSAVE / 'best.pt', weights_dir / 'best.pt')
    RESUME = True
else:
    print('Fresh conservative train.')

In [ ]:
from ultralytics import YOLO
import threading, time

stop_flag = [False]
def autosave_loop():
    while not stop_flag[0]:
        time.sleep(300)
        for src in [PROJECT/'train/weights/last.pt', PROJECT/'train/weights/best.pt']:
            if src.exists():
                try: shutil.copy2(src, AUTOSAVE / src.name)
                except: pass
        rcsv = PROJECT/'train/results.csv'
        if rcsv.exists():
            try: shutil.copy2(rcsv, AUTOSAVE / 'results.csv')
            except: pass
        print(f'[autosave] {time.strftime("%H:%M:%S")} -> {AUTOSAVE}')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()

if RESUME:
    model = YOLO(str(PROJECT / 'train/weights/last.pt'))
    train_kwargs = {'resume': True}
else:
    model = YOLO('yolo11m.pt')
    train_kwargs = {
        'data': str(data_yaml),
        'epochs': 80,
        'batch': 4,              # A100 batch=8까지 가능, 안전하게 4
        'imgsz': 1280,
        'cache': 'disk',
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 1e-4,             # 핵심: 5e-4 → 1e-4 (안정 수렴)
        'lrf': 0.01,
        'cos_lr': True,
        'patience': 25,
        'warmup_epochs': 3,
        'close_mosaic': 25,
        'freeze': 10,            # 백본 freeze (안정화)
        'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4,
        'degrees': 5.0, 'translate': 0.1, 'scale': 0.5,
        'shear': 2.0, 'perspective': 0.001,
        'flipud': 0.0, 'fliplr': 0.5,
        'mosaic': 0.5, 'mixup': 0.0, 'copy_paste': 0.3,  # 약한 augmentation
        'erasing': 0.0,
        'multi_scale': 0.2,
        'save_period': 5,
        'plots': True,
        'project': str(PROJECT.parent),
        'name': PROJECT.name,
        'exist_ok': True,
    }

results = model.train(**train_kwargs)
stop_flag[0] = True
print('Train done.')

In [ ]:
best_path = PROJECT / 'train/weights/best.pt'
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=4)
print(f'\n=== M1-conservative 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'0.9? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

OUT = DRIVE_DIR / 'm1_conservative_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm1_yolo_structural.onnx')
shutil.copy2(best_path, OUT / 'm1_conservative_best.pt')
if (best_path.parent.parent / 'results.csv').exists():
    shutil.copy2(best_path.parent.parent / 'results.csv', OUT / 'results.csv')
for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)
print('Saved:', OUT)